# Run13 timing capture on Kaggle

This notebook captures one ordinary Run13 iteration and one milestone iteration, then combines them using Run13's actual 95/5 milestone cadence for V4 architecture and throughput comparisons.

Before running: enable a GPU accelerator (preferably P100), enable Internet if cloning from GitHub, attach the Kaggle Dataset containing `latest-training.pth.tar` and `latest.examples.npz`, and push the timing changes to the configured repository branch. Large transient checkpoints stay in `/kaggle/tmp`; only the small result bundle is written to `/kaggle/working`.

## 0. P100-compatible PyTorch

Run this before any cell imports PyTorch.

In [ ]:
import subprocess, sys

try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
    ).strip()
except Exception as exc:
    gpu_name = ""
    print("Could not query the GPU:", exc)

print("GPU:", gpu_name or "none")
if "P100" in gpu_name:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--upgrade", "--force-reinstall", "--no-cache-dir",
        "torch==2.10.0", "--index-url",
        "https://download.pytorch.org/whl/cu126",
    ])
    print("Installed the P100-compatible PyTorch wheel.")

## 1. Configuration

Change the repository or branch if the timing work is not on `main`. Leave `RUN13_SOURCE` empty to auto-detect a unique attached Dataset containing both required artifacts. For a private repository, add a Kaggle secret named `GITHUB_PAT`.

In [ ]:
REPO_URL = "https://github.com/Luminous9/alpha-zero-custom.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/alpha-zero-general"
RUN13_SOURCE = ""  # Or: /kaggle/input/<dataset>/<artifact-folder>
MILESTONE_INTERVAL = 20
RUN_SMOKE_FIRST = True


## 2. Clone the exact code under test and install lightweight dependencies

In [ ]:
import base64, os, subprocess
from pathlib import Path

git_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("GITHUB_PAT")
except Exception:
    github_token = ""

if github_token:
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.extraHeader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
        "GIT_TERMINAL_PROMPT": "0",
    })

if Path(REPO_DIR, ".git").is_dir():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True, env=git_env)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True, env=git_env)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True, env=git_env)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "coloredlogs", "tqdm"], check=True)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3. Validate GPU, code, and Run13 artifacts

In [ ]:
import datetime, json, torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a Kaggle GPU accelerator before continuing.")
print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))

required_scripts = [
    Path(REPO_DIR, "benchmark_santorini_run13_timing.py"),
    Path(REPO_DIR, "summarize_santorini_run13_timing.py"),
]
for path in required_scripts:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Push the timing changes and select the correct branch.")

if RUN13_SOURCE:
    source_dir = Path(RUN13_SOURCE)
else:
    candidates = []
    for checkpoint in Path("/kaggle/input").rglob("latest-training.pth.tar"):
        if checkpoint.with_name("latest.examples.npz").is_file():
            candidates.append(checkpoint.parent)
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one Run13 artifact directory, found {candidates}. Set RUN13_SOURCE explicitly.")
    source_dir = candidates[0]

for filename in ("latest-training.pth.tar", "latest.examples.npz"):
    path = source_dir / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    print(filename, round(path.stat().st_size / 1024**3, 3), "GiB")

run_tag = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
scratch_root = Path("/kaggle/tmp") / f"run13-timing-{run_tag}"
durable_root = Path("/kaggle/working") / f"run13-timing-results-{run_tag}"
scratch_root.mkdir(parents=True, exist_ok=False)
durable_root.mkdir(parents=True, exist_ok=False)
print("Run13 source:", source_dir)
print("Scratch output:", scratch_root)
print("Durable results:", durable_root)

## 4. Optional wiring smoke

This uses two games and four simulations. It does not contribute to the baseline, but catches missing dependencies or incompatible artifacts before the two long captures.

In [ ]:
if RUN_SMOKE_FIRST:
    subprocess.run([
        sys.executable, "benchmark_santorini_run13_timing.py",
        "--source", str(source_dir),
        "--output", str(scratch_root / "smoke"),
        "--profile", "ordinary",
        "--smoke",
    ], cwd=REPO_DIR, check=True)
else:
    print("Smoke skipped.")

## 5. Capture one ordinary Run13 iteration

This is the first of the two representative captures: 240 games, 96 simulations, full replay, and up to 1,500 training steps.

In [ ]:
ordinary_dir = scratch_root / "ordinary"
subprocess.run([
    sys.executable, "benchmark_santorini_run13_timing.py",
    "--source", str(source_dir),
    "--output", str(ordinary_dir),
    "--profile", "ordinary",
], cwd=REPO_DIR, check=True)

## 6. Capture one milestone Run13 iteration

This repeats the representative workload with milestone arena and telemetry enabled.

In [ ]:
milestone_dir = scratch_root / "milestone"
subprocess.run([
    sys.executable, "benchmark_santorini_run13_timing.py",
    "--source", str(source_dir),
    "--output", str(milestone_dir),
    "--profile", "milestone",
], cwd=REPO_DIR, check=True)

## 7. Combine, inspect, and preserve the baseline

In [ ]:
import shutil

ordinary_summary = ordinary_dir / "timing-summary.json"
milestone_summary = milestone_dir / "timing-summary.json"
baseline_path = durable_root / "run13_timing_baseline.json"
subprocess.run([
    sys.executable, "summarize_santorini_run13_timing.py",
    "--ordinary", str(ordinary_summary),
    "--milestone", str(milestone_summary),
    "--milestone-interval", str(MILESTONE_INTERVAL),
    "--json-out", str(baseline_path),
], cwd=REPO_DIR, check=True)

shutil.copy2(ordinary_summary, durable_root / "ordinary_timing_summary.json")
shutil.copy2(milestone_summary, durable_root / "milestone_timing_summary.json")
for profile, profile_dir in (("ordinary", ordinary_dir), ("milestone", milestone_dir)):
    telemetry = profile_dir / "telemetry" / "telemetry.jsonl"
    if telemetry.is_file():
        shutil.copy2(telemetry, durable_root / f"{profile}_telemetry.jsonl")

baseline = json.loads(baseline_path.read_text())
print(f"Amortized total: {baseline['wall_total_seconds']:.1f} seconds")
for phase, values in baseline["phases"].items():
    print(f"{phase:18s} {values['seconds']:9.1f}s  {values['fraction']:7.1%}")

## 8. Create the downloadable bundle

Download the ZIP shown by this cell, or save a Kaggle notebook version with outputs. Send the ZIP back for inclusion in the P0b results.

In [ ]:
from IPython.display import FileLink, display

archive = shutil.make_archive(str(durable_root), "zip", root_dir=durable_root)
print("Result files:")
for path in sorted(durable_root.iterdir()):
    print(" -", path.name, path.stat().st_size, "bytes")
display(FileLink(archive))